## 🧶🏖️📏 Plot shoreline comparison.

This notebook makes shoreline plots comparing the ZC17 shoreline to the new one, looking at all planets on one projected panel.

In [ ]:
from shoreline import * 

## Load posterior.

The reads in a posterior containing samples from the shoreline parameters.

In [ ]:
# load the posterior and store in shoreline object

posterior = az.from_netcdf(f'cosmic-shoreline-btwm2026-posterior.nc')

In [ ]:
# define some planets we might want to annotate
planets_to_annotate=['Mercury',
                     'Venus',
                     'Earth',
                     'Mars',
                     'Jupiter',
                     'Saturn',
                     'Uranus',
                     'Neptune',
                     'Moon',
                     'Ganymede',
                     'Titan',
                     'Triton',
                     'Pluto',
                     'Eris',
                     'Ceres',
                     #'Haumea',
                     #'Makemake',
                     #'LHS 3844 b',
                     #'GJ367b',
                     #'TOI-1685b',
                     #'GJ1252b',
                     #'GJ486b',
                     #'GJ1132b',
                     #'LTT1445Ab',
                     #'TOI-1468 b',
                     #'LHS 1140 c',
                     #'LHS 1140 b',
                     #'Trappist-1b',
                     #'Trappist-1e',
                     #'Trappist-1f',
                     #'Trappist-1g',
                     #'Trappist-1h',
                     #'GJ 3929b',
                     #'55 Cnc e'
]

## Make some plots.

Let's make a few different visualizations, some static and some animated. 

### Grid of shoreline slices. 

First, we'll make a grid showing different slices of the 3D shoreline space side-by-side.

In [ ]:
unlabled_pops = get_all_planets()
del unlabled_pops['nontransit'] 
if False:
    ok = unlabled_pops['transit'].radius() < 1.8*u.R_earth 
    ok *= unlabled_pops['transit'].get_fractional_uncertainty('radius') < 0.5
    unlabled_pops['transit'] = unlabled_pops['transit'][ok]
#dict(transit=TransitingExoplanets(), solar=SolarSystem())
luminosity_pops = {}
log_L_limits = [0, -3]
log_L_centers = np.linspace(*log_L_limits, 4)
log_L_width = 0.5 
cmap = one2another('red', 'gray')
norm = plt.matplotlib.colors.Normalize(vmin=log_L_limits[1], vmax=log_L_limits[0])
#colors = {log_L:cmap((log_L - log_L_limits[0])/np.diff(log_L_limits)) for log_L in log_L_centers}
colors = {log_L:cmap(norm(log_L)) for log_L in log_L_centers}

for log_L in log_L_centers:
    for k, v in unlabled_pops.items():
        i = np.abs(v.log_relative_stellar_luminosity() - log_L) < 0.5

        if np.any(i):
            this = v[i]
            this.color = colors[log_L]
            this.c = this.color
            if 'transit' in k:
                this.s = 20      
                if log_L != 0:
                    this.label = None
            else:
                if 'major' in k:
                    this.s = 40 
                else:
                    this.s = 20
                    this.label = None
            luminosity_pops[f'{k}-logL={int(log_L)}'] = this 
            print(this, this.color)
            print(this._plotkw)
for k, v in clean_pops(luminosity_pops).items():
    v.annotate_planets = True 
    v.annotate_kw = dict(rotation=0, rotation_mode='anchor', format="    {}", names=planets_to_annotate, fontsize=4)
luminosity_pops

In [ ]:
# create a standard shoreline cube plot
m = ErrorMap(xaxis=RelativeEscapeVelocity(scale='log', lim=[3e-2, 1e2/3]), 
              yaxis=RelativeInstellation(scale='log', lim=[1e-4/3, 3e4]), invisible_fraction=0.4)
# plot the planets


v_earth = 1#SolarSystem()['Earth'].escape_velocity()[0]
v_earth

for zc_option in ['with', 'without']:
    fi, ax = plt.subplots(figsize=(5,4), dpi=300, constrained_layout=True)
    # refine the plots, including adding probabilities
    #g.refine()
    # save the figure
    s = Shoreline(posterior=posterior)
    log_f_0, p, q, ln_w = s.best_parameters()
    for log_L in log_L_centers:
        log_v = np.linspace(-2,2)
        log_f_shoreline = s.log_f_shoreline(log_f_0=log_f_0, p=p, q=q, log_v=log_v, log_L=log_L)
        if log_L == 0:
            label = 'Berta-Thompson et al. (2026)'
        else:
            label = None 
        plt.plot(10**log_v*v_earth, 10**log_f_shoreline, color=colors[log_L], zorder=-1e10, label=label)
        w = np.exp(ln_w)
        plt.fill_between(10**log_v*v_earth, 10**log_f_shoreline/w, 10**log_f_shoreline*w, color=colors[log_L], alpha=0.1, linewidth=0, zorder=-2e10,)
        y_max = 3.5e4
        x_max = np.interp(y_max,  10**log_f_shoreline, 10**log_v*v_earth)
        font_kw = dict(color=colors[log_L],  ha='center', va='bottom', fontsize=6.5)
        plt.text(x_max, y_max, f'$\sf 10^{{{log_L:.0f}}}$', **font_kw)
        if log_L == 0.0:
            plt.text(x_max*0.5, y_max, f'$\sf (L_\star/L_\odot) =$',  **font_kw)  

        if zc_option == 'with':
            zc_normalization = 15
            zc_p = 4
            zc_q = 0.6
            if log_L == 0:
                label = 'Zahnle & Catling (2017)'
            else:
                label = None
            plt.plot(10**log_v*v_earth, 10**(zc_p*log_v)*zc_normalization*(10**log_L)**zc_q, color=colors[log_L], linestyle=':', label=label, zorder=-1e10)

    m.build(luminosity_pops, ax=ax)
    plt.scatter(log_L_centers, log_L_centers, c=log_L_centers, cmap=cmap, norm=norm, visible=False)
    colorbar=    plt.colorbar(label='$\sf log_{10}(L_\star/L_\odot)$', aspect=50)
    colorbar.ax.tick_params(labelsize=font_kw['fontsize']) 
    plt.legend(frameon=False, fontsize='xx-small', title='The Cosmic Shoreline:', loc='upper left')
    plt.savefig(f'figures/shoreline-on-one-plot-comparison-{zc_option}-zc17.pdf')

        


In [ ]:
!cp figures/shoreline-on-one-plot-comparison-*-zc17.pdf paper-figures/.